# Power analysis attack on AES with custom SBox 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import serial.tools.list_ports as port_list
import sys
sys.path.append( '../AES_python' )
sys.path.append( '../sca_python' )

ports = list(port_list.comports())
for p in ports:
    print (p)

sbox_dict = {
    "sbox_aes" : "RIJANDAEL",
    "sbox_freyre_1" : "FREYRE_1",
    "sbox_freyre_2" : "FREYRE_2",
    "sbox_freyre_3" : "FREYRE_3",
    "sbox_hussain_6" : "HUSSAIN_6",
    "sbox_ozkaynak_1" : "OZKAYNAK_1",
    "sbox_azam_1" : "AZAM_1",
    "sbox_azam_2" : "AZAM_2",
    "sbox_azam_3" : "AZAM_3"
}

sbox_type = "sbox_azam_3"

project_file = "../build/sca_success_rate_" + sbox_dict[sbox_type] + "/sca_test_CW305.cwp"
bitstream = r"../rtl/fpga/bitstream/cw305_top_aes_single_round_" + sbox_dict[sbox_type] + r".bit"


Working with:
UnitInfo(driver=<picosdk.ps5000a.Ps5000alib object at 0x7fae6503d5b0>, variant=b'5244D', serial=b'KU687/0175')
/dev/ttyS4 - n/a
/dev/ttyUSB3 - JTAG+3Serial - JTAG+3Serial
/dev/ttyUSB2 - JTAG+3Serial - JTAG+3Serial
/dev/ttyUSB1 - JTAG+3Serial - JTAG+3Serial
/dev/ttyUSB0 - JTAG+3Serial - JTAG+3Serial


## Picoscope and CW305 initialization

In [ ]:
from pico_api import PS5000aWrapper
from CW305_api import CW305Wrapper

try:
    # Initialize picoscope
    ps = PS5000aWrapper()
    ps.get_unitInfo()
    ps.scope_setup()
    # Initialize CW305
    cw305 = CW305Wrapper(ps, bitstream)

except ModuleNotFoundError as e:
    print(e)

## Online phase : AES power traces capture

In [ ]:
from tqdm.notebook import tnrange
import chipwhisperer as cw
from Crypto.Cipher import AES
from AES import AES as AES_golden_model

project = cw.create_project(project_file, overwrite=True)

# Initialize key,text pair generator
ktp = cw.ktp.Basic()
key, text = ktp.next()

# Initialize the AES software emulated cipher
cipher = AES_golden_model()

print("Key: ", [ hex(subkey) for subkey in key])

# Write the key to the CW305
cw305.set_key(key)
# Number of traces to capture
N = 4000

# Dummy capture call due to bug of using AC coupling
cw305.capture_trace(text, project)

for i in tnrange(N, desc='Capturing traces'):
    response = cw305.capture_trace(text, project)

    # Sanity check with expected ciphertext
    key_py = str(key).replace(" ", "")
    key_py = key_py[14:-2]

    text_py = str(text).replace(" ", "")
    text_py = text_py[14:-2]
    text_py = [text_py[i:i+2] for i in range(0, len(text_py), 2)]
    text_py = ['0x' + element for element in text_py]
    text_py = [int(s, 16) for s in text_py]

    ciphertext = cipher.encrypt(key_py, text_py, sbox_type)
    assert (list(response) == ciphertext), "Incorrect encryption result!\nGot {}\nExp {}\n".format(list(response), ciphertext)

    key, text = ktp.next() 

project.save()
project.close()
# Disconnect CW305 and picoscope
cw305.dis()
ps.dis()

## Offline phase : attack on captured traces

In [ ]:
import os
import chipwhisperer.analyzer as cwa
import chipwhisperer as cw
import holoviews as hv
hv.extension('bokeh')
from analyzer.attack.aes.SBox_leakage_models import AES128SboxResistantLeakageModels
from analyzer.attack.aes.key_schedule import key_schedule_rounds
from analyzer.utils.sca_plots import sca_plot

sca_plt = sca_plot()
project = cw.open_project(project_file)

### Plotting power traces captured (40 samples overlapped)

In [ ]:
power_plt = sca_plt.power_traces_overlapped(ps.get_samplingInterval(), project.waves, finish=800)
power_plt.show()
# Ensure the Figures directory exists
os.makedirs("Figures", exist_ok=True)
power_plt.savefig("Figures/power_traces_overlapped"+tested_sbox+".png")

### SNR evaluation : detect leakage points

In [ ]:
from CW305_leakage_model import AES128_Round10_Model
# Last round state diff snr calculation
leak_model = AES128_Round10_Model()
snr_lr = cwa.calculate_snr(project.traces, leak_model=leak_model, db=False)
curve *= hv.Curve(snr_lr).opts(height=600, width=600, color='blue', alpha=0.5)

curve.opts(width=900, height=600, show_legend=True, legend_position='right', xlabel='Sample', ylabel='SNR', title="SNR in time") 


**CPA attack**

In [8]:
attack = cw.cpa(project, cwa.leakage_models.last_round_state_diff)
cb = cwa.get_jupyter_callback(attack)
results = attack.run(cwa.get_jupyter_callback(attack, 10))

NameError: name 'cwa' is not defined

In [34]:
from chipwhisperer.analyzer.attacks.models.aes.key_schedule import key_schedule_rounds
recv_lastroundkey = [kguess[0][0] for kguess in results.find_maximums()]
recv_key = key_schedule_rounds(recv_lastroundkey, 10, 0)
print("Recovered key: ", [hex(subkey) for subkey in recv_key])
key=list(project.keys[0])
assert (key == recv_key), "Failed to recover encryption key!\nGot {}\nExp {}\n".format(recv_key, key)
print("Key recovery : Success!")

Recovered key:  ['0xc0', '0x63', '0x8b', '0xfb', '0x25', '0xe7', '0x77', '0xb9', '0xa', '0x24', '0x1f', '0x48', '0xd4', '0x18', '0x79', '0x57']


AssertionError: Failed to recover encryption key!
Got [192, 99, 139, 251, 37, 231, 119, 185, 10, 36, 31, 72, 212, 24, 121, 87]
Exp [43, 126, 21, 22, 40, 174, 210, 166, 171, 247, 21, 136, 9, 207, 79, 60]


Graphical Results

In [88]:
import holoviews as hv
from holoviews.operation.datashader import datashade, shade, dynspread, rasterize
from holoviews.operation import decimate
import pandas as pd, numpy as np
plot_data = cwa.analyzer_plots(results)